# Transform Settlement Data

1. Read bronze `settlement` table
1. Drop unwanted columns
1. Standardise column names using snake_case 
1. Filter out rows where `txn_id` is null (business key validation)
1. Remove duplicate records
1. Write the transformed data to silver `circuits` table


In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
import pyspark.sql.functions as F

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.settlement"
silver_table = f"{catalog_name}.{silver_schema}.settlement"

In [0]:
settle_df = spark.read.table(bronze_table).filter(F.col("batch_id")==v_batch_id)

In [0]:
settle_renamed = settle_df.withColumnsRenamed({"TxnID":"txn_id", "BillerID":"biller_id", "CustomerID":"customer_id", "TxnAmount":"txn_amount", "Status":"status", "SettlementCycleID":"RRN", "Timestamp":"settle_time"})

In [0]:
settle_final = settle_renamed.filter(F.col("txn_id").isNotNull()).dropDuplicates()

In [0]:
settle_final = settle_final.withColumn("created_timestamp", F.current_timestamp()) \
                    .withColumn("updated_timestamp", F.current_timestamp())

In [0]:


if not spark.catalog.tableExists(silver_table):

    (
        settle_final.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

else:

    settle_final.createOrReplaceTempView("vw_settle_final")

    spark.sql(f"""
        MERGE INTO {silver_table} AS tgt
        USING vw_settle_final AS src
        ON tgt.txn_id = src.txn_id

        WHEN MATCHED
        AND src.batch_id >= tgt.batch_id
        THEN UPDATE SET
            tgt.biller_id          = src.biller_id,
            tgt.customer_id        = src.customer_id,
            tgt.txn_amount         = src.txn_amount,
            tgt.status             = src.status,
            tgt.RRN                = src.RRN,
            tgt.settle_time        = src.settle_time,
            tgt.ingestion_timestamp= src.ingestion_timestamp,
            tgt.source_file        = src.source_file,
            tgt.batch_id           = src.batch_id,
            tgt.updated_timestamp  = current_timestamp()

        WHEN NOT MATCHED
        THEN INSERT (
            txn_id,
            biller_id,
            customer_id,
            txn_amount,
            status,
            RRN,
            settle_time,
            ingestion_timestamp,
            source_file,
            batch_id,
            created_timestamp,
            updated_timestamp
        )
        VALUES (
            src.txn_id,
            src.biller_id,
            src.customer_id,
            src.txn_amount,
            src.status,
            src.RRN,
            src.settle_time,
            src.ingestion_timestamp,
            src.source_file,
            src.batch_id,
            src.created_timestamp,
            src.updated_timestamp
        )
    """)

In [0]:
display(spark.table(silver_table))